# AI Continuity Editor: evaluation on a free Kaggle GPU

Runs the whole project (extraction, story bible, checker, evaluation) on Kaggle instead of a laptop, for two models:

1. `qwen3:4b-instruct`: the same small model as on the laptop, now with the improved extraction and checker;
2. `qwen3:14b`: a bigger model, too large for an 8 GB laptop.

The results of both, plus the laptop's first run, end up in `/kaggle/working/results/`.

**Before running** (right-hand panel of the notebook editor):
1. **Session options → Accelerator:** `GPU T4 x2`. **Internet:** on.
2. **Add-ons → Secrets:** add `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD` (from your AuraDB credentials file) and tick them for this notebook. The password never appears in the notebook.
3. **Input:** add the dataset you uploaded with the project zip.
4. Resume the AuraDB instance at https://console.neo4j.io if it is paused.
5. **Save Version → Save & Run All (Commit).** It then runs in the background (up to 12 hours), even with the browser closed. Afterwards, download `results` from the version's **Output** tab.

In [ ]:
# The models to evaluate, in order. "think": False makes a thinking model answer directly.
RUNS = [
    {"model": "qwen3:4b-instruct", "think": None},
    {"model": "qwen3:14b", "think": False},
]

## 1. Copy the project and install Python 3.12 with the project's packages

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import time
import urllib.request


def run(*args, env=None):
    """Run a command, show its output as it comes, and stop the notebook if it fails."""
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, **(env or {})})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed: {' '.join(args)}")


def find(pattern, root="/kaggle/input"):
    """Files under `root` matching `pattern`, following Kaggle's linked input folders."""
    return [os.path.join(d, f) for d, _, files in os.walk(root, followlinks=True)
            for f in files if pathlib.PurePath(f).match(pattern)]


found = find("pyproject.toml")
if not found:  # Kaggle sometimes keeps an uploaded zip as it is: unpack it here
    for archive in find("*.zip"):
        shutil.unpack_archive(archive, "/kaggle/working/upload")
    found = find("pyproject.toml", "/kaggle/working/upload")
if not found:
    for d, _, files in os.walk("/kaggle/input", followlinks=True):
        print(d, files[:5])
    raise RuntimeError("Project not found. Is the uploaded dataset added under Input (right-hand panel)? "
                       "The folders printed above are everything this notebook can see.")
PROJECT = pathlib.Path("/kaggle/working/GenAI")
shutil.copytree(pathlib.Path(found[0]).parent, PROJECT, dirs_exist_ok=True)
os.chdir(PROJECT)

RESULTS = pathlib.Path("/kaggle/working/results")
if (PROJECT / "data" / "results" / "evaluation.json").exists():  # the laptop's first run, for comparison
    shutil.copytree(PROJECT / "data" / "results", RESULTS / "laptop_first_run", dirs_exist_ok=True)

run("pip", "install", "-q", "uv")
run("uv", "venv", "--python", "3.12", ".venv")
run("uv", "pip", "install", "--python", ".venv/bin/python", "-q", "-r", "requirements.txt", "-e", ".")
PY = str(PROJECT / ".venv" / "bin" / "python")
run(PY, "--version")

## 2. Install and start Ollama on the GPU

In [ ]:
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")
run("bash", "-c", "(apt-get update -qq && apt-get install -y -qq zstd) > /dev/null 2>&1 || true")
run("bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh")

ollama_log = open("/kaggle/working/ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama is running:", urllib.request.urlopen("http://localhost:11434/api/version", timeout=2).read().decode())
        break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start: see /kaggle/working/ollama.log")

## 3. Connect to Neo4j (password from Kaggle Secrets) and run the offline tests

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for key in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
    os.environ[key] = secrets.get_secret(key)  # passed to the scripts as environment variables, never saved

run(PY, "scripts/init_db.py")
run(PY, "-m", "pytest", "-q", "-m", "not integration")
run(PY, "scripts/prepare_novel.py")

## 4. Evaluate each model

For each model: download it, check it answers, then run `scripts/evaluate.py` (extract both versions of the book, check them, score the injected contradictions). The small model takes about 1-2 hours, the bigger one about 4-5.

In [ ]:
for r in RUNS:
    env = {"LLM_MODEL": r["model"]}
    if r["think"] is not None:
        env["LLM_THINK"] = str(r["think"]).lower()
    started = time.time()
    run("ollama", "pull", r["model"])
    run(PY, "scripts/check_llm.py", env=env)
    run(PY, "scripts/evaluate.py", env=env)
    shutil.copytree(PROJECT / "data" / "results", RESULTS / r["model"].replace(":", "_"), dirs_exist_ok=True)
    print(f"{r['model']} finished in {(time.time() - started) / 3600:.1f} hours", flush=True)

## 5. Compare the runs

In [ ]:
import pandas as pd

rows = []
for folder in sorted(RESULTS.iterdir()):
    path = folder / "evaluation.json"
    if not path.exists():
        continue
    e = json.loads(path.read_text())
    p, r = e["precision"] or 0.0, e["recall"]
    rows.append({
        "run": folder.name, "model": e["meta"]["model"], "checker": e["meta"]["checker"],
        "recall": f"{e['detected']}/{e['contradictions']} ({r:.0%})",
        "precision": f"{e['true_positive_flags']}/{e['flags']} ({p:.0%})",
        "F1": f"{(2 * p * r / (p + r) if p + r else 0):.0%}",
        "high-tier precision": f"{e['by_confidence']['high']['precision']:.0%}",
        "tier agreement": "n/a" if e["tier_agreement"] is None else f"{e['tier_agreement']:.0%}",
    })
pd.DataFrame(rows).set_index("run")